# Compare Baseline Model with XGBoost Model

Tests the full pipeline with baseline model and XGBoost model to compare them

## 1. Load data


In [4]:
import sys
sys.path.append('../src')  # adjust path if needed

from data import load_features, split_data

features_df = load_features()
train, val, test = split_data(features_df)

Loading existing features from cache...
Train: 2758 | Val: 586 | Test: 566


### 1.2 Check data

In [5]:

print(f"Train: {len(train):4d} shots | {train['is_goal'].mean():.1%} goals")
print(f"Val:   {len(val):4d} shots | {val['is_goal'].mean():.1%} goals")
print(f"Test:  {len(test):4d} shots | {test['is_goal'].mean():.1%} goals")

# Verify no match appears in more than one split (data leakage check)
train_ids = set(train['match_id'].unique())
val_ids   = set(val['match_id'].unique())
test_ids  = set(test['match_id'].unique())

assert len(train_ids & val_ids)   == 0, "❌ Overlap between Train and Val!"
assert len(train_ids & test_ids)  == 0, "❌ Overlap between Train and Test!"
assert len(val_ids   & test_ids)  == 0, "❌ Overlap between Val and Test!"
print("\n✅ No overlap between splits — split is clean")

Train: 2758 shots | 8.6% goals
Val:    586 shots | 8.0% goals
Test:   566 shots | 9.9% goals

✅ No overlap between splits — split is clean


## 2. Train and evaluate baseline model

In [6]:
from models import train_baseline, train_xgboost, make_X_tabular
from evaluate import evaluate

# Baseline
model_baseline = train_baseline(train)
X_val_baseline = val[['distance', 'angle']].values
y_val = val['is_goal'].values
evaluate(model_baseline, X_val_baseline, y_val, df=val, label='Logistic Regression')

Model                     AUC   Brier
--------------------------------------
Logistic Regression     0.788  0.0666
StatsBomb xG            0.830  0.0584


## 3. Train and evaluate XGBoost model

In [7]:
# XGBoost
model_xgboost = train_xgboost(train)
X_val_xgboost = make_X_tabular(val)
evaluate(model_xgboost, X_val_xgboost, y_val, df=val, label='XGBoost')

Model                     AUC   Brier
--------------------------------------
XGBoost                 0.795  0.0686
StatsBomb xG            0.830  0.0584
